In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="text-align:right;padding:8px 0;">
<button id="toggle-code-btn" style="
  padding:8px 18px;font-size:14px;cursor:pointer;
  background:#4a90d9;color:white;border:none;border-radius:5px;
  box-shadow:0 2px 4px rgba(0,0,0,0.2);
" onclick="
  var cells = document.querySelectorAll('.jp-Cell-inputWrapper');
  var btn = document.getElementById('toggle-code-btn');
  var show = btn.dataset.show !== 'true';
  btn.dataset.show = show;
  btn.textContent = show ? '\u9690\u85cf\u4ee3\u7801 \ud83d\udd12' : '\u663e\u793a\u4ee3\u7801 \ud83d\udd13';
  cells.forEach(function(c){ c.style.display = show ? '' : 'none'; });
">\u663e\u793a\u4ee3\u7801 \ud83d\udd13</button>
</div>
'''))


# 第12章 软件开发 - 12.2 程序设计
# Chapter 12 Software Development - 12.2 Program Design

---

**Cambridge International AS & A Level Computer Science (9618)**

在开发生命周期中，**设计**阶段是连接「需求分析」和「编码」的桥梁。
好的设计让编码变得简单，差的设计让编码变成噩梦。

本节我们学习两种重要的设计工具：
- **结构图** (Structure Chart): 展示模块的层次结构和数据流
- **状态转换图** (State-Transition Diagram): 展示系统的状态变化

## 1. 结构图 Structure Charts

### 1.1 什么是结构图？

结构图是一种**自顶向下** (top-down) 的图形化工具，用于展示：
- 一个系统由哪些**模块** (modules) 组成
- 模块之间的**层次关系** (hierarchy)
- 模块之间的**数据流** (data flow)
- 模块之间的**控制信号** (control signals)

**类比：** 想象一个公司的组织架构图——
- 最上面是 CEO（主模块）
- 下面是各部门经理（子模块）
- 再下面是各个员工（更细的子模块）
- 部门之间会传递信息（数据流）和指令（控制信号）

### 1.2 结构图的符号 Symbols

| 符号 | 名称 | 含义 |
|---|---|---|
| 矩形框 | Module (模块) | 一个处理单元/子程序 |
| 空心箭头 (→○) | Data Couple (数据偶) | 数据从一个模块传递到另一个模块 |
| 实心箭头 (→●) | Flag/Control Couple (控制偶) | 控制信号（如「成功/失败」） |
| 菱形 / 星号 (*) | Selection (选择) | 条件分支（只执行某些子模块） |
| 弧形箭头 / 循环符号 | Iteration (迭代) | 循环调用子模块 |

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 WenQuanYi Zen Hei 字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('WenQuanYi Zen Hei')
if 'WenQuanYi Zen Hei' in test_font or 'wqy' in test_font.lower():
    print('✅ 中文字体 WenQuanYi Zen Hei 已加载')
else:
    print(f'⚠️ WenQuanYi Zen Hei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 WenQuanYi Zen Hei 字体文件')

import matplotlib.patches as mpatches
import matplotlib.lines as mlines

fig, ax = plt.subplots(figsize=(12, 4))
ax.set_xlim(0, 16)
ax.set_ylim(0, 5)
ax.axis('off')

# Module box
rect = mpatches.FancyBboxPatch((0.5, 2), 2.5, 1.5, boxstyle='round,pad=0.05',
                               facecolor='#d5f4e6', edgecolor='black', lw=1.5)
ax.add_patch(rect)
ax.text(1.75, 2.75, 'Module\n(模块)', ha='center', va='center', fontsize=9, fontweight='bold')

# Data couple
ax.annotate('', xy=(5.5, 3), xytext=(4, 3),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='blue'))
ax.plot(4.5, 3.3, 'o', color='white', markersize=8, markeredgecolor='blue', markeredgewidth=1.5)
ax.text(4.75, 4, 'Data Couple\n数据偶（空心圆）', ha='center', fontsize=9, color='blue')

# Flag/Control couple
ax.annotate('', xy=(9, 3), xytext=(7.5, 3),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='red'))
ax.plot(8.0, 3.3, 'o', color='red', markersize=8, markeredgecolor='red')
ax.text(8.25, 4, 'Flag/Control Couple\n控制偶（实心圆）', ha='center', fontsize=9, color='red')

# Selection
diamond = mpatches.RegularPolygon((11.5, 3), numVertices=4, radius=0.4,
                                  facecolor='lightyellow', edgecolor='black', lw=1.5)
ax.add_patch(diamond)
ax.text(11.5, 4, 'Selection\n选择（菱形）', ha='center', fontsize=9)

# Iteration
from matplotlib.patches import Arc
arc = Arc((14, 3), 1.0, 0.8, angle=0, theta1=30, theta2=330, lw=2)
ax.add_patch(arc)
ax.annotate('', xy=(14.4, 3.3), xytext=(14.35, 3.15),
            arrowprops=dict(arrowstyle='->', lw=1.5))
ax.text(14, 4, 'Iteration\n迭代（弧形箭头）', ha='center', fontsize=9)

ax.set_title('Structure Chart Symbols (结构图符号)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 1.3 如何阅读结构图 How to Read a Structure Chart

阅读结构图的规则：

1. **从上到下：** 最上面是主模块（Main Module），向下逐层分解
2. **从左到右：** 同一层的模块，通常从左到右执行
3. **数据流方向：** 空心箭头指示数据从哪个模块流向哪个模块
4. **控制流方向：** 实心箭头指示控制信号的方向
5. **条件和循环：** 注意菱形（选择）和弧形（迭代）标记

> **关键理解：** 结构图体现了**分解** (Decomposition) 的思想——
> 把一个复杂的大问题拆成若干简单的小问题。
> 这是计算机科学中最重要的思维方式之一！

### 1.4 实例：学生成绩报告系统 Worked Example: Student Report Card System

**问题描述：** 设计一个系统，能够：
1. 从文件中读取学生信息和成绩
2. 计算每个学生的平均分
3. 根据平均分确定等级（A/B/C/D/F）
4. 打印成绩报告

让我们一步步构建结构图。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 12)
ax.axis('off')

def draw_module(ax, x, y, w, h, label, color='#d5f4e6'):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.05',
                                   facecolor=color, edgecolor='black', lw=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=8, fontweight='bold')

# Level 0: Main module
draw_module(ax, 4.5, 10, 5, 1.2, 'Generate Report Card\n生成成绩报告', '#87ceeb')

# Level 1: Three main sub-modules
draw_module(ax, 0.5, 7, 3.5, 1.2, 'Read Data\n读取数据', '#98fb98')
draw_module(ax, 5, 7, 4, 1.2, 'Process Data\n处理数据', '#ffd700')
draw_module(ax, 10, 7, 3.5, 1.2, 'Print Report\n打印报告', '#ffb6c1')

# Level 2: Sub-modules of Read Data
draw_module(ax, 0, 4, 2, 1.2, 'Open File\n打开文件')
draw_module(ax, 2.5, 4, 2.5, 1.2, 'Read Records\n读取记录')

# Level 2: Sub-modules of Process Data
draw_module(ax, 5, 4, 2, 1.2, 'Calc Average\n计算平均分')
draw_module(ax, 7.5, 4, 2, 1.2, 'Assign Grade\n确定等级')

# Level 2: Sub-modules of Print Report
draw_module(ax, 10, 4, 1.8, 1.2, 'Print Header\n打印表头')
draw_module(ax, 12, 4, 1.8, 1.2, 'Print Details\n打印详情')

# Level 3: Sub-module of Assign Grade
draw_module(ax, 7.5, 1.5, 2, 1.2, 'Check\nBoundary\n检查边界')

# Connecting lines - Level 0 to Level 1
ax.plot([7, 2.25], [10, 8.2], 'k-', lw=1.5)
ax.plot([7, 7], [10, 8.2], 'k-', lw=1.5)
ax.plot([7, 11.75], [10, 8.2], 'k-', lw=1.5)

# Connecting lines - Level 1 to Level 2
ax.plot([2.25, 1], [7, 5.2], 'k-', lw=1.5)
ax.plot([2.25, 3.75], [7, 5.2], 'k-', lw=1.5)
ax.plot([7, 6], [7, 5.2], 'k-', lw=1.5)
ax.plot([7, 8.5], [7, 5.2], 'k-', lw=1.5)
ax.plot([11.75, 10.9], [7, 5.2], 'k-', lw=1.5)
ax.plot([11.75, 12.9], [7, 5.2], 'k-', lw=1.5)

# Connecting line - Level 2 to Level 3
ax.plot([8.5, 8.5], [4, 2.7], 'k-', lw=1.5)

# Data couples (open circles with arrows)
# Read Data sends student_data to Process Data
ax.annotate('', xy=(5.0, 7.8), xytext=(4.0, 7.8),
            arrowprops=dict(arrowstyle='->', color='blue', lw=1.2))
ax.plot(4.3, 8.05, 'o', color='white', markersize=6, markeredgecolor='blue', mew=1.5)
ax.text(4.5, 8.5, 'student_data', fontsize=7, color='blue', fontstyle='italic')

# Process Data sends results to Print Report
ax.annotate('', xy=(10.0, 7.8), xytext=(9.0, 7.8),
            arrowprops=dict(arrowstyle='->', color='blue', lw=1.2))
ax.plot(9.3, 8.05, 'o', color='white', markersize=6, markeredgecolor='blue', mew=1.5)
ax.text(9.3, 8.5, 'results', fontsize=7, color='blue', fontstyle='italic')

# Iteration mark on Read Records
from matplotlib.patches import Arc
arc = Arc((2.8, 5.8), 0.8, 0.5, angle=0, theta1=30, theta2=330, lw=1.5)
ax.add_patch(arc)

# Selection mark on Assign Grade
diamond = mpatches.RegularPolygon((7.8, 5.8), numVertices=4, radius=0.2,
                                  facecolor='lightyellow', edgecolor='black', lw=1)
ax.add_patch(diamond)

ax.set_title('Structure Chart: Student Report Card System\n结构图：学生成绩报告系统',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.5 阅读上面的结构图

让我们分析这个结构图：

**层次结构：**
- **顶层 (Level 0):** `Generate Report Card` — 整个系统的入口
- **第一层 (Level 1):** 三个主要子任务
  - `Read Data` — 读取学生数据
  - `Process Data` — 处理数据（计算和评级）
  - `Print Report` — 输出报告
- **第二层 (Level 2):** 进一步分解
  - `Read Data` → `Open File` + `Read Records`（循环读取）
  - `Process Data` → `Calc Average` + `Assign Grade`（条件选择）
  - `Print Report` → `Print Header` + `Print Details`

**数据流：**
- `Read Data` 将 `student_data` 传递给 `Process Data`（蓝色空心箭头）
- `Process Data` 将 `results` 传递给 `Print Report`

**特殊标记：**
- `Read Records` 有迭代标记（弧形）→ 需要循环读取多条记录
- `Assign Grade` 有选择标记（菱形）→ 根据条件执行不同的评级逻辑

### 1.6 从结构图推导伪代码 Deriving Pseudocode from Structure Chart

结构图直接对应程序的模块划分。每个方框变成一个过程/函数：

In [ ]:
# 从结构图推导的伪代码实现
# Pseudocode derived from the structure chart

def open_file(filename):
    """Level 2: Open File 打开文件"""
    print(f'Opening file: {filename}')
    # 模拟数据
    return [
        {'name': 'Alice', 'scores': [85, 92, 78]},
        {'name': 'Bob', 'scores': [65, 70, 58]},
        {'name': 'Charlie', 'scores': [95, 98, 92]},
    ]

def read_records(raw_data):
    """Level 2: Read Records 读取记录 (iteration - 循环)"""
    records = []
    for record in raw_data:  # 迭代标记对应 for 循环
        records.append(record)
        print(f'  Read record: {record["name"]}')
    return records

def read_data(filename):
    """Level 1: Read Data 读取数据"""
    raw_data = open_file(filename)
    records = read_records(raw_data)
    return records  # student_data 传递给 process_data

def calc_average(scores):
    """Level 2: Calc Average 计算平均分"""
    return sum(scores) / len(scores)

def check_boundary(average):
    """Level 3: Check Boundary 检查边界"""
    if average >= 90: return 'A'
    elif average >= 80: return 'B'
    elif average >= 70: return 'C'
    elif average >= 60: return 'D'
    else: return 'F'

def assign_grade(average):
    """Level 2: Assign Grade 确定等级 (selection - 选择)"""
    return check_boundary(average)  # 选择标记对应 if-else

def process_data(student_data):
    """Level 1: Process Data 处理数据"""
    results = []
    for student in student_data:
        avg = calc_average(student['scores'])
        grade = assign_grade(avg)
        results.append({'name': student['name'], 'average': avg, 'grade': grade})
    return results  # results 传递给 print_report

def print_header():
    """Level 2: Print Header 打印表头"""
    print('\n' + '='*45)
    print('  STUDENT REPORT CARD (学生成绩报告)')
    print('='*45)
    print(f'{"Name":<12} {"Average":<10} {"Grade":<6}')
    print('-'*45)

def print_details(results):
    """Level 2: Print Details 打印详情"""
    for r in results:
        print(f'{r["name"]:<12} {r["average"]:<10.1f} {r["grade"]:<6}')
    print('='*45)

def print_report(results):
    """Level 1: Print Report 打印报告"""
    print_header()
    print_details(results)

def generate_report_card(filename):
    """Level 0: Generate Report Card 生成成绩报告 (Main Module)"""
    student_data = read_data(filename)
    results = process_data(student_data)
    print_report(results)

# 运行主模块
generate_report_card('students.dat')

### 1.7 互动：一步步构建结构图 Interactive: Build a Structure Chart Step by Step

让我们通过一个新的问题，学习如何**从问题描述构建**结构图。

**问题：** 设计一个ATM（自动取款机）系统，功能包括：
- 验证用户身份（输入卡号和PIN码）
- 查询余额
- 取款（检查余额是否足够）
- 存款
- 打印交易凭条

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 16)
ax.set_ylim(0, 11)
ax.axis('off')

def draw_mod(ax, x, y, w, h, label, color='#d5f4e6'):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.05',
                                   facecolor=color, edgecolor='black', lw=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=8, fontweight='bold')

# Level 0
draw_mod(ax, 5.5, 9, 5, 1.2, 'ATM System\nATM 系统', '#87ceeb')

# Level 1
draw_mod(ax, 0.5, 6, 2.5, 1.2, 'Authenticate\n身份验证', '#ff9999')
draw_mod(ax, 3.5, 6, 2.5, 1.2, 'Check Balance\n查询余额', '#98fb98')
draw_mod(ax, 6.5, 6, 2.5, 1.2, 'Withdraw\n取款', '#ffd700')
draw_mod(ax, 9.5, 6, 2.5, 1.2, 'Deposit\n存款', '#dda0dd')
draw_mod(ax, 12.5, 6, 2.5, 1.2, 'Print Receipt\n打印凭条', '#ffa500')

# Level 2 - Authenticate sub-modules
draw_mod(ax, 0, 3, 1.8, 1.2, 'Read Card\n读卡')
draw_mod(ax, 2.2, 3, 1.8, 1.2, 'Verify PIN\n验证PIN')

# Level 2 - Withdraw sub-modules
draw_mod(ax, 5.5, 3, 2, 1.2, 'Check Funds\n检查余额')
draw_mod(ax, 8, 3, 2, 1.2, 'Dispense Cash\n出钞')

# Lines Level 0 -> Level 1
cx = 8.0
for target_x in [1.75, 4.75, 7.75, 10.75, 13.75]:
    ax.plot([cx, target_x], [9, 7.2], 'k-', lw=1.2)

# Lines Level 1 -> Level 2 (Authenticate)
ax.plot([1.75, 0.9], [6, 4.2], 'k-', lw=1.2)
ax.plot([1.75, 3.1], [6, 4.2], 'k-', lw=1.2)

# Lines Level 1 -> Level 2 (Withdraw)
ax.plot([7.75, 6.5], [6, 4.2], 'k-', lw=1.2)
ax.plot([7.75, 9.0], [6, 4.2], 'k-', lw=1.2)

# Selection mark on Level 1 (user chooses which operation)
diamond = mpatches.RegularPolygon((8.0, 8.3), numVertices=4, radius=0.2,
                                  facecolor='lightyellow', edgecolor='black', lw=1)
ax.add_patch(diamond)

# Data couples
ax.plot(2.7, 4.8, 'o', color='white', markersize=5, markeredgecolor='blue', mew=1.5)
ax.text(2.2, 5.2, 'PIN', fontsize=7, color='blue', fontstyle='italic')

# Flag couple (verified/not verified)
ax.plot(2.2, 7.6, 'o', color='red', markersize=5, markeredgecolor='red')
ax.text(2.5, 7.9, 'valid?', fontsize=7, color='red', fontstyle='italic')

ax.set_title('Structure Chart: ATM System (结构图：ATM系统)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. 状态转换图 State-Transition Diagrams

### 2.1 什么是状态转换图？

状态转换图 (State-Transition Diagram, STD) 用于描述系统在不同**状态** (State) 之间
如何**转换** (Transition)。

**核心概念：**
- **状态 (State):** 系统在某一时刻的「情况」或「模式」
- **转换 (Transition):** 从一个状态到另一个状态的变化
- **事件 (Event):** 触发转换的条件或动作
- **动作 (Action):** 转换发生时执行的操作

**类比：** 想想你自己的状态——
- 状态：睡觉 → 醒来 → 吃饭 → 学习 → 休息 → 睡觉
- 事件：闹钟响（触发「睡觉→醒来」）、饿了（触发「醒来→吃饭」）

**图中的符号：**

| 符号 | 含义 |
|---|---|
| 圆形/圆角矩形 | 状态 (State) |
| 箭头 | 转换 (Transition) |
| 箭头上的文字 | 事件/条件 [/ 动作] |
| 实心圆点 (●) | 初始状态 (Start State) |
| 带圈的实心圆点 (◉) | 终止状态 (End State) |

### 2.2 实例：交通灯控制器 Traffic Light Controller

交通灯是状态转换图最经典的例子。让我们把它画出来并用 Python 模拟！

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xlim(0, 12)
ax.set_ylim(0, 10)
ax.axis('off')

# States
states = {
    'Red':    (3, 8, '#e74c3c'),
    'Red+Amber': (9, 8, '#f39c12'),
    'Green':  (9, 3, '#2ecc71'),
    'Amber':  (3, 3, '#f39c12'),
}

for name, (x, y, color) in states.items():
    circle = plt.Circle((x, y), 1.0, facecolor=color, edgecolor='black', lw=2, alpha=0.8)
    ax.add_patch(circle)
    ax.text(x, y, name, ha='center', va='center', fontsize=10, fontweight='bold', color='white')

# Start state indicator
ax.plot(1, 8, 'ko', markersize=12)
ax.annotate('', xy=(2, 8), xytext=(1.2, 8),
            arrowprops=dict(arrowstyle='->', lw=2, color='black'))
ax.text(0.5, 8.5, 'Start', fontsize=9, fontstyle='italic')

# Transitions
# Red -> Red+Amber
ax.annotate('', xy=(8, 8), xytext=(4, 8),
            arrowprops=dict(arrowstyle='->', lw=2, color='#333'))
ax.text(6, 8.5, 'timer expires\n(定时器到期)', ha='center', fontsize=8, color='#555')

# Red+Amber -> Green
ax.annotate('', xy=(9, 4), xytext=(9, 7),
            arrowprops=dict(arrowstyle='->', lw=2, color='#333'))
ax.text(9.8, 5.5, 'timer expires', ha='left', fontsize=8, color='#555')

# Green -> Amber
ax.annotate('', xy=(4, 3), xytext=(8, 3),
            arrowprops=dict(arrowstyle='->', lw=2, color='#333'))
ax.text(6, 2.3, 'timer expires', ha='center', fontsize=8, color='#555')

# Amber -> Red
ax.annotate('', xy=(3, 7), xytext=(3, 4),
            arrowprops=dict(arrowstyle='->', lw=2, color='#333'))
ax.text(1.8, 5.5, 'timer expires', ha='right', fontsize=8, color='#555')

ax.set_title('State-Transition Diagram: Traffic Light Controller\n状态转换图：交通灯控制器',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 交通灯状态机 Python 模拟
# Traffic Light State Machine Simulation
import time

class TrafficLight:
    def __init__(self):
        self.state = 'RED'
        self.transitions = {
            'RED': 'RED_AMBER',
            'RED_AMBER': 'GREEN',
            'GREEN': 'AMBER',
            'AMBER': 'RED'
        }
        self.durations = {
            'RED': 5,       # 红灯 5 秒
            'RED_AMBER': 2, # 红黄灯 2 秒
            'GREEN': 5,     # 绿灯 5 秒
            'AMBER': 3      # 黄灯 3 秒
        }
        self.symbols = {
            'RED': '\U0001F534',
            'RED_AMBER': '\U0001F534\U0001F7E1',
            'GREEN': '\U0001F7E2',
            'AMBER': '\U0001F7E1'
        }
    
    def next_state(self):
        old_state = self.state
        self.state = self.transitions[self.state]
        return old_state, self.state
    
    def simulate(self, cycles=2):
        print('Traffic Light State Machine Simulation')
        print('='*50)
        step = 0
        for _ in range(cycles * 4):  # 4 states per cycle
            symbol = self.symbols[self.state]
            duration = self.durations[self.state]
            print(f'Step {step}: State = {self.state:<12} {symbol}  (duration: {duration}s)')
            old, new = self.next_state()
            print(f'         Transition: {old} -> {new}')
            print()
            step += 1

light = TrafficLight()
light.simulate(cycles=2)

### 2.3 实例：自动售货机 Vending Machine

自动售货机是另一个经典的状态转换例子。
想象一个只卖一种饮料（价格 5 元）的简单售货机：

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')

# States
vm_states = {
    'Idle':       (2, 8, '#95a5a6'),
    'Coin 1':     (7, 8, '#3498db'),
    'Coin 2':     (12, 8, '#3498db'),
    'Coin 3':     (12, 4, '#3498db'),
    'Coin 4':     (7, 4, '#3498db'),
    'Dispense':   (2, 4, '#2ecc71'),
}

for name, (x, y, color) in vm_states.items():
    circle = plt.Circle((x, y), 0.9, facecolor=color, edgecolor='black', lw=2, alpha=0.8)
    ax.add_patch(circle)
    label = name
    if 'Coin' in name:
        n = int(name.split()[-1])
        label = f'{n} Yuan\n({n}元)'
    ax.text(x, y, label, ha='center', va='center', fontsize=9, fontweight='bold', color='white')

# Start
ax.plot(0.3, 8, 'ko', markersize=10)
ax.annotate('', xy=(1.1, 8), xytext=(0.5, 8),
            arrowprops=dict(arrowstyle='->', lw=2))

# Transitions
transitions = [
    ((2, 8), (7, 8), 'insert 1 Yuan'),
    ((7, 8), (12, 8), 'insert 1 Yuan'),
    ((12, 8), (12, 4), 'insert 1 Yuan'),
    ((12, 4), (7, 4), 'insert 1 Yuan'),
    ((7, 4), (2, 4), 'insert 1 Yuan\n/ dispense drink'),
    ((2, 4), (2, 8), 'complete\n/ return to idle'),
]

for (x1, y1), (x2, y2), label in transitions:
    dx = x2 - x1
    dy = y2 - y1
    dist = (dx**2 + dy**2)**0.5
    # Shorten arrows to not overlap circles
    ratio = 0.9 / dist
    sx = x1 + dx * ratio
    sy = y1 + dy * ratio
    ex = x2 - dx * ratio
    ey = y2 - dy * ratio
    ax.annotate('', xy=(ex, ey), xytext=(sx, sy),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#333'))
    mx = (x1 + x2) / 2
    my = (y1 + y2) / 2
    offset_x = -0.3 if dx == 0 else 0
    offset_y = 0.3 if dy == 0 else 0
    ax.text(mx + offset_x, my + offset_y, label, fontsize=7, ha='center', color='#555',
            fontstyle='italic')

# Cancel transition from any Coin state back to Idle
ax.annotate('', xy=(2, 8.9), xytext=(7, 8.9),
            arrowprops=dict(arrowstyle='->', lw=1.2, color='red',
                           connectionstyle='arc3,rad=0.3', linestyle='dashed'))
ax.text(4.5, 9.7, 'cancel / refund (取消/退款)', fontsize=8, color='red',
        ha='center', fontstyle='italic')

ax.set_title('State-Transition Diagram: Vending Machine (状态转换图：自动售货机)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 售货机状态机 Python 实现
class VendingMachine:
    def __init__(self, price=5):
        self.price = price
        self.balance = 0
        self.state = 'IDLE'
    
    def insert_coin(self, amount=1):
        if self.state == 'IDLE':
            self.state = 'COLLECTING'
        
        self.balance += amount
        print(f'  Inserted {amount} Yuan. Balance: {self.balance}/{self.price}')
        
        if self.balance >= self.price:
            self.dispense()
    
    def dispense(self):
        self.state = 'DISPENSING'
        change = self.balance - self.price
        print(f'  >> Dispensing drink! (出饮料！)')
        if change > 0:
            print(f'  >> Returning change: {change} Yuan (找零 {change} 元)')
        self.balance = 0
        self.state = 'IDLE'
        print(f'  >> Back to IDLE state')
    
    def cancel(self):
        if self.balance > 0:
            print(f'  >> Refunding {self.balance} Yuan (退款 {self.balance} 元)')
            self.balance = 0
        self.state = 'IDLE'
        print(f'  >> Cancelled. Back to IDLE.')

# 模拟
print('=== Vending Machine Simulation ===')
vm = VendingMachine(price=5)

print('\nScenario 1: Insert 5 coins')
for i in range(5):
    vm.insert_coin(1)

print('\nScenario 2: Insert 2 coins then cancel')
vm.insert_coin(1)
vm.insert_coin(1)
vm.cancel()

print('\nScenario 3: Insert a 5 Yuan coin directly')
vm.insert_coin(5)

## 3. 从设计到伪代码 From Design to Pseudocode

### 3.1 从状态转换图推导伪代码

**规则：**
1. 每个**状态**变成一个 `CASE` 或 `IF` 分支
2. 每个**转换**变成一个条件判断
3. 整个图变成一个**循环** + **状态变量**

以交通灯为例：

```
state <- RED
WHILE system is running
    CASE OF state
        RED:       WAIT red_duration
                   state <- RED_AMBER
        RED_AMBER: WAIT red_amber_duration
                   state <- GREEN
        GREEN:     WAIT green_duration
                   state <- AMBER
        AMBER:     WAIT amber_duration
                   state <- RED
    ENDCASE
ENDWHILE
```

### 3.2 从结构图推导伪代码

**规则：**
1. 每个**模块**变成一个 `PROCEDURE` 或 `FUNCTION`
2. **层次关系**变成函数调用
3. **数据偶**变成参数传递
4. **迭代标记**变成循环
5. **选择标记**变成条件判断

```
PROCEDURE GenerateReportCard(filename)
    studentData <- ReadData(filename)
    results <- ProcessData(studentData)
    CALL PrintReport(results)
ENDPROCEDURE

FUNCTION ReadData(filename) RETURNS ARRAY
    CALL OpenFile(filename)
    records <- ReadRecords()  // iteration
    RETURN records
ENDFUNCTION
```

## 4. 练习题 Practice Exercises

### 练习 1: 绘制结构图

为一个「在线图书馆系统」绘制结构图。该系统需要：
- 用户登录
- 搜索图书（按书名或作者）
- 借阅图书（检查是否可借）
- 归还图书
- 查看借阅历史

提示：先确定主模块，然后分解为子模块，标注数据流和控制信号。

### 练习 2: 绘制状态转换图

为一个「简单的电梯系统」绘制状态转换图。
- 电梯有三层：Ground (G), First (1), Second (2)
- 状态：停在某层、上升中、下降中、开门、关门
- 事件：按下楼层按钮、到达目标楼层、门打开超时

### 练习 3: 伪代码转换

将下面的状态转换图描述转换为伪代码：

```
一个简单的密码锁：
- 初始状态: LOCKED（锁定）
- 输入正确密码 → UNLOCKED（解锁）
- 输入错误密码 → ERROR（错误），3秒后回到 LOCKED
- 错误3次 → BLOCKED（封锁），需要管理员解锁
- 在 UNLOCKED 状态按锁定按钮 → LOCKED
```

In [ ]:
# 练习 3 参考答案：密码锁状态机
class PasswordLock:
    def __init__(self, correct_password='1234'):
        self.correct_password = correct_password
        self.state = 'LOCKED'
        self.error_count = 0
        self.max_errors = 3
    
    def enter_password(self, password):
        print(f'  Input: {password}')
        
        if self.state == 'BLOCKED':
            print('  >> BLOCKED! Contact admin. (已封锁！请联系管理员)')
            return
        
        if self.state == 'UNLOCKED':
            print('  >> Already unlocked! (已解锁！)')
            return
        
        if password == self.correct_password:
            self.state = 'UNLOCKED'
            self.error_count = 0
            print('  >> Correct! State: LOCKED -> UNLOCKED')
        else:
            self.error_count += 1
            print(f'  >> Wrong! Errors: {self.error_count}/{self.max_errors}')
            if self.error_count >= self.max_errors:
                self.state = 'BLOCKED'
                print('  >> TOO MANY ERRORS! State -> BLOCKED')
            else:
                self.state = 'ERROR'
                print('  >> State: LOCKED -> ERROR -> LOCKED')
                self.state = 'LOCKED'  # auto-return
    
    def lock(self):
        if self.state == 'UNLOCKED':
            self.state = 'LOCKED'
            print('  >> Locked! State: UNLOCKED -> LOCKED')
    
    def admin_reset(self):
        self.state = 'LOCKED'
        self.error_count = 0
        print('  >> Admin reset! State -> LOCKED')

print('=== Password Lock State Machine ===')
lock = PasswordLock('1234')

print('\nTest 1: Correct password')
lock.enter_password('1234')
lock.lock()

print('\nTest 2: Wrong passwords until blocked')
lock.enter_password('0000')
lock.enter_password('1111')
lock.enter_password('2222')
lock.enter_password('1234')  # blocked!

print('\nTest 3: Admin reset')
lock.admin_reset()
lock.enter_password('1234')

---

> **本节小结 (Summary):**
> - **结构图** (Structure Chart) 展示系统的模块层次和数据流
>   - 矩形 = 模块，空心箭头 = 数据偶，实心箭头 = 控制偶
>   - 菱形 = 选择，弧形 = 迭代
> - **状态转换图** (State-Transition Diagram) 展示系统的状态变化
>   - 圆形 = 状态，箭头 = 转换，标注 = 事件/条件
> - 两种图都可以系统地转换为伪代码
> - 好的设计文档让编码阶段事半功倍